## 1. Introduction
Automated Machine Learning (AutoML) simplifies the end-to-end ML workflow by:
- **Feature preprocessing**
- **Model selection**
- **Hyperparameter optimization**
- **Ensembling**

In this tutorial, we'll focus solely on **H2O AutoML**, a scalable AutoML framework supporting both regression and classification.

## 2. Setup & Installation

In [1]:
!pip install --quiet jedi
!pip install --quiet h2o
!pip install --quiet 'thinc<8.3.6'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.4/266.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 94.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.14 requires confection<2.0.0,>=1.3.2, but you have confection 0.1.5 which is incompatible.
spacy 3.8.14 requires thinc<8.4.0,>=8.3.12, but you have thinc 8.3.4 which is incompatible.
weasel 1.0.0 requires confection>=1.0.0, but you have confection 0.1.5 which is incompatible.


## Regression on California Housing

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## 3. Regression Example: California Housing

In [3]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
# Initialize H20
h2o.init(max_mem_size='2G', nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.19" 2026-04-21; OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu); OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpl31i6gv2
  JVM stdout: /tmp/tmpl31i6gv2/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpl31i6gv2/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 7 days
H2O_cluster_name:,H2O_from_python_unknownUser_2odrco
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.981 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [5]:
# Load the data
data = fetch_california_housing(as_frame=True)

In [10]:
X = data.data
y = data.target.rename('target')

In [11]:
# Create the H2O Frame
df = h2o.H2OFrame(pd.concat([X,y],axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [14]:
# split into train/test
train, test = df.split_frame(ratios=[0.8],seed=42)

## Run H2O AutoML For Regression

In [18]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name='california_regression'
)

In [21]:
aml_reg

In [22]:
aml_reg.train(x=X.columns.to_list(), y='target', training_frame=train)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_4_AutoML_1_20260729_142306


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    99                 99                          277091                 10           10           10            41            451           218.404

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.07397325656727911
RMSE: 0.2719802503257895
MAE: 0.18538668936475916
RMSLE: 0.0835524573825608
Mean Residual Deviance: 0.07397325656727911

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.20714453019161605
RMSE: 0.4551313329047079
MAE: 0.29676202261609314
RMSLE: 0.13681969786908746
Mean Residual Deviance: 0.20714453019161605

Cross-Validation Metrics Summary: 
                        mean      sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  --------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan       0            nan           nan           nan           nan           nan
loglikelihood           nan       0            nan           nan           nan           nan           nan
mae                     0.296732  0.00322323   0.299136      0.292047      0.294765      0.298175      0.299539
mean_residual_deviance  0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
mse                     0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
r2                      0.844695  0.00317222   0.840185      0.845816      0.847472      0.842682      0.847321
residual_deviance       0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
rmse                    0.455183  0.00668183   0.46659       0.452661      0.450643      0.455466      0.450555
rmsle                   0.136859  0.000486415  0.137507      0.136554      0.136472      0.136504      0.137256

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse        training_mae         training_deviance
---  -------------------  ----------  -----------------  -------------------  -------------------  -------------------
     2026-07-29 14:25:14  11.457 sec  0.0                1.1550860447424607   0.9126682923030268   1.334223770758782
     2026-07-29 14:25:14  11.648 sec  5.0                0.8099859863741191   0.6333660448978525   0.6560772981224546
     2026-07-29 14:25:14  11.828 sec  10.0               0.617830894441751    0.4722230808231802   0.38171501412669406
     2026-07-29 14:25:15  12.002 sec  15.0               0.5109753197147372   0.3784870497227261   0.26109577735757794
     2026-07-29 14:25:15  12.175 sec  20.0               0.44111368507007787  0.3156783019546691   0.19458128315610382
     2026-07-29 14:25:15  12.342 sec  25.0               0.4067439923962636   0.284500376285093    0.1654406753504517
     2026-07-29 14:25:15  12.501 sec  30.0               0.382227147884688    0.2624152441257563   0.14609759258006313
     2026-07-29 14:25:15  12.676 sec  35.0               0.3651895115355381   0.24789120218978403  0.1333633793355649
     2026-07-29 14:25:15  12.831 sec  40.0               0.34783585389151034  0.23473247741922482  0.12098978125243612
     2026-07-29 14:25:16  12.921 sec  45.0               0.3358994029907933   0.22632615955146831  0.11282840892957134
---  ---                  ---         ---                ---                  ---                  ---
     2026-07-29 14:25:16  13.113 sec  55.0               0.3176689889023286   0.21408124998465275  0.10091358651022775
     2026-07-29 14:25:16  13.202 sec  60.0              

In [23]:
# show leaderboard
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

                         model_id      rmse       mse       mae     rmsle  \
0  GBM_4_AutoML_1_20260729_142306  0.455131  0.207145  0.296762  0.136820   
1  GBM_2_AutoML_1_20260729_142306  0.455992  0.207929  0.300417  0.137703   
2  GBM_3_AutoML_1_20260729_142306  0.458074  0.209832  0.301015  0.137618   
3  GBM_1_AutoML_1_20260729_142306  0.459199  0.210863  0.302926  0.138295   
4  GBM_5_AutoML_1_20260729_142306  0.460024  0.211622  0.305412  0.139356   

   mean_residual_deviance  
0                0.207145  
1                0.207929  
2                0.209832  
3                0.210863  
4                0.211622  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [24]:
# Evaluate on test data
perf_reg = aml_reg.leader.model_performance(test)
print(f"H2O Regression : {perf_reg.r2():.4f}")
print(f"H2O Regression RMSE: {perf_reg.rmse():.4f}")

H2O Regression : 0.8523
H2O Regression RMSE: 0.4417


## 4. Classification Example: Breast Cancer Dataset

In [25]:
from sklearn.datasets import load_breast_cancer

In [26]:
# Load and prepare dataset
data_cls = load_breast_cancer(as_frame=True)
Xc = data_cls.data
yc = data_cls.target.rename('target')

In [27]:
df_cls = h2o.H2OFrame(pd.concat([Xc, yc], axis=1))
train_cls, test_cls = df_cls.split_frame(ratios=[0.7], seed=42)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


### 4.1 Run H2O AutoML for Classification

In [28]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    balance_classes=True,
    project_name="breast_cancer_classification"
)
aml_cls.train(x=Xc.columns.tolist(), y='target', training_frame=train_cls)

AutoML progress: |
14:37:13.454: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
14:37:15.805: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


14:37:16.166: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
14:37:17.472: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

██
14:37:19.87: _response param, We have det

key,value
Stacking strategy,cross_validation
Number of base models (used / total),4/5
# GBM base models (used / total),1/1
# XGBoost base models (used / total),1/1
# DRF base models (used / total),1/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [29]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                                            model_id      rmse       mse  \
0  StackedEnsemble_BestOfFamily_1_AutoML_2_202607...  0.178552  0.031881   
1  StackedEnsemble_AllModels_1_AutoML_2_20260729_...  0.183572  0.033699   
2       GBM_grid_1_AutoML_2_20260729_143713_model_22  0.185965  0.034583   
3       GBM_grid_1_AutoML_2_20260729_143713_model_11  0.186716  0.034863   
4       GBM_grid_1_AutoML_2_20260729_143713_model_20  0.188528  0.035543   

        mae     rmsle  mean_residual_deviance  
0  0.099521  0.128031                0.031881  
1  0.095885  0.131159                0.033699  
2  0.096905  0.131893                0.034583  
3  0.089201  0.134120                0.034863  
4  0.090711  0.133354                0.035543  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [30]:
# Evaluate on test
perf_cls = aml_cls.leader.model_performance(test_cls)
print(f"H2O Classification: {perf_cls}")

H2O Classification: ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 0.03687836387225242
RMSE: 0.19203740227427682
MAE: 0.102186978783904
RMSLE: 0.12966219676564422
Mean Residual Deviance: 0.03687836387225242
R^2: 0.836282180057665
Null degrees of freedom: 177
Residual degrees of freedom: 173
Null deviance: 40.43221198186829
Residual deviance: 6.564348769260931
AIC: -70.28106575860501


MSE (Mean Squared Error):
- Measures the average squared difference between predicted and actual values. Lower values indicate better performance.


RMSE (Root Mean Squared Error):
- The square root of MSE, providing an error metric in the same units as the target variable. Lower values are better.


MAE (Mean Absolute Error):
-Measures the average absolute difference between predicted and actual values. Lower values indicate better performance.


RMSLE (Root Mean Squared Logarithmic Error):
- Similar to RMSE but uses the logarithm of the values, making it more robust to outliers. Lower values are better.


Mean Residual Deviance:
- Measures the goodness of fit of the model. Lower values indicate a better fit.


R² (Coefficient of Determination):
- Represents the proportion of variance explained by the model. Values closer to 1 indicate better performance.


Null Degrees of Freedom:
- The number of observations minus 1.


Residual Degrees of Freedom:
- The number of observations minus the number of parameters estimated.


Null Deviance:
- The deviance of the null model (model with no predictors).


Residual Deviance:
- The deviance of the fitted model. Lower values indicate a better fit.


AIC (Akaike Information Criterion):
- A measure of model quality, balancing goodness of fit and model complexity. Lower values indicate a better model.

## 5. Interpreting Results
- **Leaderboard** displays model ranking by default metric.
- Use `model_performance` to compute custom metrics (RMSE, R², AUC, accuracy, etc.).
- Top models are automatically ensembled by H2O's Stacked Ensemble.

## 6. AutoML Best Practices
- **Set time and model limits** (`max_runtime_secs`, `max_models`) to control cost and runtime.
- **Use cross-validation** (`nfolds`) for robust performance estimates.
- **Balance classes** for imbalanced classification tasks.
- **Review variable importance** on the leader model: `aml.leader.varimp()`.
- **Save and deploy** the best model: `h2o.save_model(aml.leader, path='best_model')`.

## EXERCISE

## 7. Exercise: Custom Dataset AutoML

**Task:** Apply H2O AutoML to a custom dataset.

1. Load any tabular dataset (CSV or from `sklearn.datasets`).
2. Decide whether it's a regression or classification task.
3. Initialize H2O and convert to `H2OFrame`.
4. Split into appropriate train/test ratios.
5. Run `H2OAutoML` with:
   - `max_runtime_secs=300`
   - `max_models=15`
   - `nfolds=5`
   - `balance_classes=True` (if classification)
6. Display the leaderboard and evaluate on the test set using relevant metrics.
7. Save the leaderboard to a pandas DataFrame and export it as `leaderboard.csv`

In [59]:
from sklearn.datasets import load_digits

In [60]:
h2o.init(max_mem_size='2G', nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,1 hour 13 mins
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 7 days
H2O_cluster_name:,H2O_from_python_unknownUser_2odrco
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.766 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [61]:
# Loading the data :
data = load_digits(as_frame=True)

In [62]:
X = data.data
y = data.target.rename('target')

In [63]:
# Create a dataframe
df = h2o.H2OFrame(pd.concat([X,y],axis=1))
# IMPORTANT: Convert target to categorical
df["target"] = df["target"].asfactor()

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [64]:
# train test split
train, test = df.split_frame(ratios=[0.8],seed=42)

## Running H2O AutoML For Classification

In [65]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name='load_digits_v1'
)
aml_cls.train(x=X.columns.tolist(), y='target',training_frame=train)

AutoML progress: |
15:26:50.61: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

██
15:26:57.929: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

████████
15:27:33.70: GLM_1_AutoML_5_20260729_152650 [GLM def_1] failed: java.lang.ArrayIndexOutOfBoundsException
15:27:33.73: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

████████
15:28:15.198: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

██
15:28:21.314: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

██
15:28:31.549: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

███████
15:29:08.288: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

█████████
15:29:48.402: _train param, Dropping bad and constant columns: [pixel_4_7, pixel_0_0, pixel_4_0]

██████████
15:30:41.348: _train param, Dropping b

Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_2_AutoML_5_20260729_152650


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    85                 850                         427967                 5            7            6.98824       9             64            35.36

ModelMetricsMultinomial: gbm
** Reported on train data. **

MSE: 6.675404132987873e-09
RMSE: 8.170314640812723e-05
LogLoss: 2.3915645816922977e-05
Mean Per-Class Error: 0.0
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0    1    2    3    4    5    6    7    8    9    Error    Rate
---  ---  ---  ---  ---  ---  ---  ---  ---  ---  -------  ---------
128  0    0    0    0    0    0    0    0    0    0        0 / 128
0    139  0    0    0    0    0    0    0    0    0        0 / 139
0    0    143  0    0    0    0    0    0    0    0        0 / 143
0    0    0    152  0    0    0    0    0    0    0        0 / 152
0    0    0    0    134  0    0    0    0    0    0        0 / 134
0    0    0    0    0    147  0    0    0    0    0        0 / 147
0    0    0    0    0    0    147  0    0    0    0        0 / 147
0    0    0    0    0    0    0    148  0    0    0        0 / 148
0    0    0    0    0    0    0    0    147  0    0        0 / 147
0    0    0    0    0    0    0    0    0    151  0        0 / 151
128  139  143  152  134  147  147  148  147  151  0        0 / 1,436

Top-10 Hit Ratios: 
k    hit_ratio
---  -----------
1    1
2    1
3    1
4    1
5    1
6    1
7    1
8    1
9    1
10   1

ModelMetricsMultinomial: gbm
** Reported on cross-validation data. **

MSE: 0.0248016289783031
RMSE: 0.157485329406593
LogLoss: 0.100371944659986
Mean Per-Class Error: 0.02608190871848236
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0    1    2    3    4    5    6    7    8    9    Error       Rate
---  ---  ---  ---  ---  ---  ---  ---  ---  ---  ----------  ----------
125  0    0    0    1    0    1    0    1    0    0.0234375   3 / 128
0    137  1    1    0    0    0    0    0    0    0.0143885   2 / 139
0    1    142  0    0    0    0    0    0    0    0.00699301  1 / 143
0    1    2    143  0    1    0    2    2    1    0.0592105   9 / 152
0    0    0    0    132  0    0    1    0    1    0.0149254   2 / 134
0    0    0    0    0    143  0    0    0    4    0.0272109   4 / 147
1    1    0    0    0    1    143  0    1    0    0.0272109   4 / 147
0    0    0    0    0    0    0    146  0    2    0.0135135   2 / 148
0    1    0    1    0    0    0    0    141  4    0.0408163   6 / 147
0    1    0    0    0    1    0    1    2    146  0.0331126   5 / 151
126  142  145  145  133  146  144  150  147  158  0.0264624   38 / 1,436

Top-10 Hit Ratios: 
k    hit_ratio
---  -----------
1    0.973538
2    0.986072
3    0.993036
4    0.996518
5    0.997214
6    0.997911
7    0.999304
8    0.999304
9    0.999304
10   1

Cross-Validation Metrics Summary: 
                         mean       sd           cv_1_valid    cv_2_valid    cv_3_valid 

In [66]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                         model_id  mean_per_class_error   logloss      rmse  \
0  GBM_2_AutoML_5_20260729_152650              0.026082  0.100372  0.157485   
1  GBM_3_AutoML_5_20260729_152650              0.026164  0.094702  0.152912   
2  GBM_4_AutoML_5_20260729_152650              0.029581  0.103549  0.163112   
3  GBM_5_AutoML_5_20260729_152650              0.030327  0.115218  0.174425   
4  GBM_1_AutoML_5_20260729_152650              0.030862  0.100457  0.160241   

        mse  
0  0.024802  
1  0.023382  
2  0.026606  
3  0.030424  
4  0.025677  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [67]:
# Evaluate on test :
perf_cls = aml_cls.leader.model_performance(test)
print(f"H2O classification : {perf_cls}")

H2O classification : ModelMetricsMultinomial: gbm
** Reported on test data. **

MSE: 0.01148807437427613
RMSE: 0.10718243500814921
LogLoss: 0.03901831429286081
Mean Per-Class Error: 0.01878166217167555
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0    1    2    3    4    5    6    7    8    9    Error      Rate
---  ---  ---  ---  ---  ---  ---  ---  ---  ---  ---------  -------
50   0    0    0    0    0    0    0    0    0    0          0 / 50
0    43   0    0    0    0    0    0    0    0    0          0 / 43
0    0    34   0    0    0    0    0    0    0    0          0 / 34
0    0    0    31   0    0    0    0    0    0    0

In [68]:
# Save AutoML leaderboard as pandas DataFrame
leaderboard_df = aml_cls.leaderboard.as_data_frame()

# Display leaderboard
print(leaderboard_df.head())

# Export to CSV
leaderboard_df.to_csv("leaderboard.csv", index=False)

print("Leaderboard saved successfully as leaderboard.csv")

                         model_id  mean_per_class_error   logloss      rmse  \
0  GBM_2_AutoML_5_20260729_152650              0.026082  0.100372  0.157485   
1  GBM_3_AutoML_5_20260729_152650              0.026164  0.094702  0.152912   
2  GBM_4_AutoML_5_20260729_152650              0.029581  0.103549  0.163112   
3  GBM_5_AutoML_5_20260729_152650              0.030327  0.115218  0.174425   
4  GBM_1_AutoML_5_20260729_152650              0.030862  0.100457  0.160241   

        mse  
0  0.024802  
1  0.023382  
2  0.026606  
3  0.030424  
4  0.025677  
Leaderboard saved successfully as leaderboard.csv


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
